# GReLU neuron on XX vs XXX thermal states — the minimal pipeline

This notebook holds the **entire** minimal pipeline, with every script pasted in full:
`neuron.jl` → `data.jl` → `run.jl` → `plots.jl`. Nothing is hidden in an `include`.

1. **The neuron** — GReLU, the Hamiltonian H(θ), the exact losses and gradients (Algorithms 9 and 8), and firing (Algorithm 5).
2. **The data** — 840 thermal states of random 10-qubit XX and XXX chains, rebuilt exactly, then split by chain.
3. **Training** — Adam on each loss.
4. **Testing** — exact test accuracy, then classification by firing with finite copies.
5. **Figures**.

The write-up is [REPORT.md](REPORT.md). The paper is He, Liu & Wilde, *Fermi–Dirac machines as quantizations of neurons* (arXiv:2605.24386).

**Running.** Use the "Julia 1.12" kernel, from anywhere inside the repository. The whole notebook takes about 15 minutes, almost all of it training. It needs HDF5, Optimisers, Plots and SpecialFunctions.

In [ ]:
# Paths: find the repository root (the folder that contains data/xx_xxx_thermal_states)
function find_root(dir=pwd())
  while !isdir(joinpath(dir, "data", "xx_xxx_thermal_states"))
    parent = dirname(dir)
    parent == dir && error("start the notebook from inside the repository")
    dir = parent
  end
  return dir
end
const ROOT = find_root()
const HERE = joinpath(ROOT, "3_xx_xxx_grelu_minimal")
println("repository: ", ROOT)

## 1. The neuron (`neuron.jl`)

The header comment below is the whole idea in one place: what the neuron computes, how the two algorithms train it, and how firing works.

### Activation
GReLU_T(x) = x Φ(x/T) + T φ(x/T): a ReLU smoothed by a Gaussian of width T.

In [ ]:
# neuron.jl — the quantized GReLU neuron (He, Liu & Wilde, "Fermi–Dirac machines
# as quantizations of neurons", arXiv:2605.24386, Sec. IV.B).
#
# A classical neuron computes ReLU(w·x).  The quantum version takes a density
# matrix rho and a Hamiltonian built from trainable weights,
#
#     H(theta) = sum_j theta_j H_j        (H_j = Pauli strings: Z, ZZ, XX, YY)
#
# and its output is   Tr[ GReLU_T(H(theta)) rho ],   where
#
#     GReLU_T(x) = x Phi(x/T) + T phi(x/T)      (a smoothed ReLU; Phi, phi =
#                                               standard normal cdf and pdf).
#
# Training.  Algorithms 8 and 9 of the paper are quantum circuits whose
# measured outcomes average to the gradient of a loss.  Here we use that
# average directly: the EXACT gradient, which is what the circuits give in the
# limit of infinitely many runs.  (Folder 2 simulates the circuits shot by shot.)
#
#   Algorithm 9 — margin loss   L = mean_m Tr[GReLU_T(-y_m H) rho_m]
#                 classify by the sign of Tr[H rho]
#   Algorithm 8 — squared loss  L = mean_m (Tr[GReLU_T(H) rho_m] - t_m)^2,  t = 0 (XX) or 1 (XXX)
#                 classify by output > 1/2; needs a constant (identity) term
#
# Classification.  Algorithm 5 ("firing" the neuron) uses ONE copy of rho per
# run and returns a random number whose average is the neuron output.  In the
# eigenbasis of H, one run is: pick an eigenvalue E with probability <E|rho|E>,
# then return ReLU(E + T z) with z ~ N(0, 1).

using LinearAlgebra, Random, SparseArrays, Statistics
using SpecialFunctions: erf

# ------------------------------------------------------------- activation ---

Phi(x) = (1 + erf(x / sqrt(2))) / 2
phi(x) = exp(-x^2 / 2) / sqrt(2pi)
grelu(x, T) = x * Phi(x / T) + T * phi(x / T)

### Pauli strings and H(θ)
Each term Hⱼ is a sparse 1024 × 1024 Pauli string. The model has 37 terms: 10 Z, 9 ZZ, 9 XX and 9 YY. Algorithm 8 adds an identity term as a bias.

In [ ]:
# ---------------------------------------------------- Pauli strings and H ---

const PAULI = Dict('I' => ComplexF64[1 0; 0 1], 'X' => ComplexF64[0 1; 1 0],
                   'Y' => ComplexF64[0 -im; im 0], 'Z' => ComplexF64[1 0; 0 -1])

"Sparse matrix of a Pauli string on n qubits, e.g. ops = [(3, 'X'), (4, 'X')]."
function pauli(n, ops)
  factors = [sparse(PAULI['I']) for _ in 1:n]
  for (site, P) in ops
    factors[site] = sparse(PAULI[P])
  end
  return real(reduce(kron, factors))        # every string we use is real
end

"""
The neuron's terms: Z_i, Z_iZ_{i+1}, X_iX_{i+1}, Y_iY_{i+1} (37 terms at n = 10),
plus the identity first if `bias` (Algorithm 8 needs it).  Returns (names, matrices).
"""
function neuron_terms(n; bias=false)
  names, ops = String[], Vector{Tuple{Int,Char}}[]
  bias && (push!(names, "I"); push!(ops, Tuple{Int,Char}[]))
  for i in 1:n
    push!(names, "Z$i"); push!(ops, [(i, 'Z')])
  end
  for P in "ZXY", i in 1:(n - 1)
    push!(names, "$P$i$P$(i+1)"); push!(ops, [(i, P), (i + 1, P)])
  end
  return names, [pauli(n, o) for o in ops]
end

hamiltonian(terms, theta) = Matrix(sum(t .* Hj for (t, Hj) in zip(theta, terms)))

### The states, seen by H
Each state is ρ = W diag(p) Wᵀ, using its chain's eigenvectors W. To evaluate the neuron we only need the state's populations on the eigenvectors of H(θ).

In [ ]:
# ------------------------------------------------- the states, seen by H ---
#
# Every input state is rho_m = W diag(p_m) W' (its chain's eigenvectors W and
# Boltzmann weights p_m; see data.jl).  With H = V diag(E) V', the populations
# of rho_m on H's eigenvectors are (A.^2) p_m, with A = V'W.

"For each state m in `idx`: its populations <E|rho_m|E> on the eigenvectors V of H."
function populations(ds, idx, V)
  pops = zeros(size(V, 1), length(idx))
  for c in unique(ds.chain[idx])
    A2 = (V' * ds.chains[c].W) .^ 2
    for (i, m) in enumerate(idx)
      ds.chain[m] == c && (pops[:, i] = A2 * boltzmann(ds.chains[c], ds.beta[m]))
    end
  end
  return pops
end

"Class averages (1/M) sum rho_m over the XXX states and over the XX states of `idx`."
function class_averages(ds, idx)
  R = Dict(+1.0 => zeros(2^ds.n, 2^ds.n), -1.0 => zeros(2^ds.n, 2^ds.n))
  for m in idx
    ch = ds.chains[ds.chain[m]]
    R[ds.y[m]] .+= ch.W * Diagonal(boltzmann(ch, ds.beta[m]) ./ length(idx)) * ch.W'
  end
  return R[+1.0], R[-1.0]
end

### Exact losses and gradients
The Daleckii–Krein formula differentiates Tr[f(H) ρ] exactly. The result equals the average of what the Algorithm 9 and Algorithm 8 circuits measure, in the limit of infinitely many runs.

In [ ]:
# --------------------------------------------- exact losses and gradients ---
#
# Derivative of a matrix function (Daleckii–Krein):
#   d/dtheta_j Tr[f(H) R] = Tr[H_j V (G ∘ V'RV) V'],   G_ab = (f(E_a) - f(E_b)) / (E_a - E_b)

function divided_differences(E, T)
  G = similar(E, length(E), length(E))
  for (k, b) in enumerate(E), (i, a) in enumerate(E)
    G[i, k] = abs(a - b) < 1e-9 ? Phi((a + b) / 2T) :          # a ≈ b: f'(a) = Phi(a/T)
                                  (grelu(a, T) - grelu(b, T)) / (a - b)
  end
  return G
end

gradient_from(terms, V, K) = (M = V * K * V'; [dot(Hj, M) for Hj in terms])


"Algorithm 8's loss and its exact gradient, on the states `idx`."
function square_loss_grad(ds, idx, terms, theta, T)
  E, V = eigen(Symmetric(hamiltonian(terms, theta)))
  chains = unique(ds.chain[idx])
  A = Dict(c => V' * ds.chains[c].W for c in chains)     # each chain's eigenvectors, seen by H
  p = [boltzmann(ds.chains[ds.chain[m]], ds.beta[m]) for m in idx]
  out = [dot(grelu.(E, T), (A[ds.chain[m]] .^ 2) * p[i]) for (i, m) in enumerate(idx)]
  r = out .- (ds.y[idx] .+ 1) ./ 2                       # residuals vs targets {0, 1}
  # gradient of mean(r^2) = gradient of Tr[GReLU(H) Rc] with Rc = (2/M) sum_m r_m rho_m
  Rc = zeros(size(V))
  for c in chains
    q = sum(2r[i] / length(idx) .* p[i] for (i, m) in enumerate(idx) if ds.chain[m] == c)
    Rc .+= A[c] * Diagonal(q) * A[c]'
  end
  return mean(abs2, r), gradient_from(terms, V, divided_differences(E, T) .* Rc)
end

"""
Exact predictions on the states `idx` (the infinite-copy limit):
Algorithm 9 model -> score Tr[H rho];  Algorithm 8 model -> score Tr[GReLU(H) rho] - 1/2.
Predict XXX (+1) when the score is >= 0.
"""
function scores(kind, ds, idx, terms, theta, T)
  E, V = eigen(Symmetric(hamiltonian(terms, theta)))
  pops = populations(ds, idx, V)
  return kind == :margin ? pops' * E : pops' * grelu.(E, T) .- 0.5
end

### Firing (Algorithm 5)
One firing uses one copy of ρ. It picks an eigenvalue E of H with probability ⟨E|ρ|E⟩, then outputs ReLU(E + T z) with z ~ N(0, 1). Its mean is the neuron output.

In [ ]:
# ---------------------------------------------------- firing (Algorithm 5) ---

"One firing of the neuron on one copy of the state: ReLU(E + T z)."
function fire(E, cdf, T, rng; sign=1)
  i = min(searchsortedfirst(cdf, rand(rng)), length(E))
  return max(sign * E[i] + T * randn(rng), 0.0)
end

"""
Classify one state from `ncopies` firings.
Algorithm 8 model: average output > 1/2.
Algorithm 9 model: needs Tr[H rho] = Tr[GReLU(H) rho] - Tr[GReLU(-H) rho], so it
spends half the copies on the neuron built from H and half on the one from -H.
"""
function classify_by_firing(kind, E, pop, T, ncopies, rng)
  cdf = cumsum(max.(pop, 0) ./ sum(max.(pop, 0)))
  if kind == :square
    return mean(fire(E, cdf, T, rng) for _ in 1:ncopies) > 0.5 ? 1.0 : -1.0
  end
  h = max(ncopies ÷ 2, 1)
  s = mean(fire(E, cdf, T, rng) for _ in 1:h) - mean(fire(E, cdf, T, rng; sign=-1) for _ in 1:h)
  return s >= 0 ? 1.0 : -1.0
end

## 2. The data (`data.jl`)

The dataset holds 84 random chains, 42 XX and 42 XXX, each at 10 temperatures.

In [ ]:
# data.jl — load the XX vs XXX thermal states and split them into train / test.
#
# The dataset (../data/xx_xxx_thermal_states/xx_xxx_n10.h5) holds 840 thermal
# states of random 10-qubit spin chains:
#
#   XX  chain:  H = sum_i J_i (X_i X_{i+1} + Y_i Y_{i+1})                label y = -1
#   XXX chain:  H = sum_i J_i (X_i X_{i+1} + Y_i Y_{i+1} + Z_i Z_{i+1})  label y = +1
#
# 84 chains (42 of each kind), each at 10 temperatures kT.  The file also ships
# each state as an MPO, but we rebuild every state exactly from its couplings
# J instead: rho = exp(-H/kT) / Z.  We diagonalise each chain once,
# H = W diag(eps) W', and then every temperature is just a different set of
# Boltzmann weights on the same eigenvectors.

using HDF5, LinearAlgebra, Random

# (pauli() comes from the neuron cells above)

### One chain = its eigenvectors
Each chain is diagonalised once. Every temperature then reuses the same eigenvectors with different Boltzmann weights.

In [ ]:
struct Chain
  W::Matrix{Float64}     # eigenvectors of the chain Hamiltonian (columns)
  eps::Vector{Float64}   # its eigenvalues
end

"Boltzmann weights e^{-beta eps} / Z: the state's populations in the chain's eigenbasis."
function boltzmann(ch::Chain, beta)
  w = exp.(-beta .* (ch.eps .- minimum(ch.eps)))
  return w ./ sum(w)
end

"The density matrix itself (only needed for checks; training never forms it)."
density_matrix(ch::Chain, beta) = ch.W * Diagonal(boltzmann(ch, beta)) * ch.W'

### The dataset and the loader
The loader reads the couplings J, rebuilds each chain Hamiltonian and diagonalises it. It then checks every state's energy against the file.

In [ ]:
struct Dataset
  n::Int
  chains::Vector{Chain}
  chain::Vector{Int}      # which chain each state comes from
  group::Vector{String}   # "XX:3" = model XX, draw 3 (one per chain)
  model::Vector{String}
  kT::Vector{Float64}
  beta::Vector{Float64}
  y::Vector{Float64}      # -1 = XX, +1 = XXX
end

function chain_hamiltonian(n, J, xxx::Bool)
  H = zeros(2^n, 2^n)
  for i in 1:(n - 1)
    H .+= J[i] .* (pauli(n, [(i, 'X'), (i + 1, 'X')]) .+ pauli(n, [(i, 'Y'), (i + 1, 'Y')]))
    xxx && (H .+= J[i] .* pauli(n, [(i, 'Z'), (i + 1, 'Z')]))
  end
  return H
end


"Read the dataset and diagonalise every chain (~1 min)."
function load_dataset(path)
  rows = []
  n = 0
  h5open(path, "r") do f
    n = Int(read(attributes(f["meta"])["n"]))
    for name in sort(keys(f["samples"]))
      g = f["samples"][name]
      a = attributes(g)
      push!(rows, (model=String(read(a["model"])), draw=Int(read(a["draw"])),
                   kT=Float64(read(a["kT"])), beta=Float64(read(a["beta"])),
                   J=Vector{Float64}(read(g["J"])), energy=Float64(read(a["energy_ed"]))))
    end
  end

  group = ["$(r.model):$(r.draw)" for r in rows]
  groups = unique(group)
  chains = map(groups) do g
    r = rows[findfirst(==(g), group)]
    F = eigen(Symmetric(chain_hamiltonian(n, r.J, r.model == "XXX")))
    Chain(F.vectors, F.values)
  end
  chain = [findfirst(==(g), groups) for g in group]

  # sanity check: our rebuilt states have the energies the dataset generator recorded
  dE = maximum(abs(dot(boltzmann(chains[chain[m]], r.beta), chains[chain[m]].eps) - r.energy)
               for (m, r) in enumerate(rows))
  dE < 1e-8 || error("rebuilt states disagree with the file (max |dE| = $dE)")

  model = [r.model for r in rows]
  return Dataset(n, chains, chain, group, model, [r.kT for r in rows], [r.beta for r in rows],
                 [m == "XXX" ? 1.0 : -1.0 for m in model])
end

### Train / test split by chain
Whole chains are held out, so the test chains are ones the neuron has never seen.

In [ ]:
"""
Hold out whole chains: 8 XX chains and 8 XXX chains (all 10 temperatures of each)
go to the test set, the other 68 chains to training.  Splitting by chain, not by
state, means the test set contains only chains the neuron has never seen.
"""
function split_by_chain(ds::Dataset; test_per_class=8, seed=20260923)
  rng = MersenneTwister(seed)
  test = falses(length(ds.y))
  for cls in ("XX", "XXX")
    gs = sort(unique(ds.group[ds.model .== cls]); by=g -> parse(Int, split(g, ':')[2]))
    for g in shuffle(rng, gs)[1:test_per_class]
      test[ds.group .== g] .= true
    end
  end
  return findall(.!test), findall(test)
end

## 3. The pipeline (`run.jl`)

### Settings
These settings were chosen by cross-validation in folder 2.

In [ ]:
# run.jl — the whole minimal pipeline:  data -> split -> train -> test -> fire -> figures.
# (plots.jl redraws the figures from results/ alone, without retraining.)
#
#   julia 3_xx_xxx_grelu_minimal/run.jl          (from the repository root; ~15 min)
#
# Writes results/*.csv and figures/*.png next to this file.

using Optimisers, Printf

const DATA = joinpath(ROOT, "data", "xx_xxx_thermal_states", "xx_xxx_n10.h5")
const RESULTS = mkpath(joinpath(HERE, "results"))
const FIGURES = mkpath(joinpath(HERE, "figures"))

# Settings (chosen by cross-validation in folder 2)
const T = 1.0                                       # activation width of GReLU_T
const L2 = Dict(:margin => 1e-3, :square => 1e-4)   # weight decay
const STEPS, LR, SEED = 200, 0.05, 20260923
const NAME = Dict(:margin => "Alg 9 (margin loss)", :square => "Alg 8 (squared loss)")

writecsv(path, header, rows) = open(io -> (println(io, join(header, ","));
                                           foreach(r -> println(io, join(r, ",")), rows)), path, "w")

### Load and split
This step takes about a minute, mostly diagonalising 84 dense 1024 × 1024 Hamiltonians.

In [ ]:
# ------------------------------------------------------------------ 1. data ---

println("loading data ...")
ds = load_dataset(DATA)
train, test = split_by_chain(ds)
@printf("  %d states from %d chains;  train %d states (%d chains),  test %d states (%d chains)\n",
        length(ds.y), length(ds.chains), length(train), length(unique(ds.chain[train])),
        length(test), length(unique(ds.chain[test])))

accuracy(score, idx) = mean((score .>= 0) .== (ds.y[idx] .> 0))

### Train both neurons
Adam runs on loss + (λ/2)|θ|² with the exact gradient. Test accuracy is printed every 10 steps. This is the slow step (~12 min).

In [ ]:
# ----------------------------------------------------------------- 2. train ---

"""
Adam on loss + L2/2 |theta|^2 with the exact gradient.  `kind` is :margin
(Algorithm 9) or :square (Algorithm 8).  Returns the weights and a history.
"""
function train_neuron(kind)
  names, terms = neuron_terms(ds.n; bias=(kind == :square))
  theta = 0.1 .* randn(MersenneTwister(SEED), length(terms))
  opt = Optimisers.setup(Optimisers.Adam(LR), theta)
  kind == :margin && ((Rp, Rm) = class_averages(ds, train))
  history = []
  for step in 0:STEPS
    L, g = kind == :margin ? margin_loss_grad(terms, theta, T, Rp, Rm) :
                             square_loss_grad(ds, train, terms, theta, T)
    if step % 10 == 0
      acc = accuracy(scores(kind, ds, test, terms, theta, T), test)
      push!(history, (step, L, acc))
      @printf("  %-22s step %3d   train loss %.4f   test accuracy %.3f\n", NAME[kind], step, L, acc)
    end
    step == STEPS && break
    opt, theta = Optimisers.update!(opt, theta, g .+ L2[kind] .* theta)
  end
  return (; names, terms, theta, history)
end

models = Dict(kind => train_neuron(kind) for kind in (:margin, :square))

writecsv(joinpath(RESULTS, "history.csv"), ["model", "step", "train_loss", "test_accuracy"],
         [(k, h...) for k in (:margin, :square) for h in models[k].history])
writecsv(joinpath(RESULTS, "weights.csv"), ["model", "term", "theta"],
         [(k, n, t) for k in (:margin, :square) for (n, t) in zip(models[k].names, models[k].theta)])

### Test: exact outputs
Each model's decision score is computed for every test state. Exact outputs correspond to infinitely many copies of each state.

In [ ]:
# ------------------------------------------------------------------ 3. test ---

println("\ntest set (exact outputs = infinitely many copies of each state):")
rows = []
for kind in (:margin, :square)
  m = models[kind]
  s = scores(kind, ds, test, m.terms, m.theta, T)
  @printf("  %-22s accuracy %.3f   ||theta||_1 = %.1f\n", NAME[kind], accuracy(s, test), norm(m.theta, 1))
  append!(rows, [(kind, ds.group[i], ds.kT[i], Int(ds.y[i]), s[j]) for (j, i) in enumerate(test)])
end
writecsv(joinpath(RESULTS, "test_scores.csv"), ["model", "chain", "kT", "y", "score"], rows)

### Classify by firing
Accuracy is measured against the number of copies of each test state, averaged over 10 repeats. The Algorithm 9 model splits its copies between the neurons built from H and from −H.

In [ ]:
# ------------------------------------------------- 4. firing (Algorithm 5) ---

println("\nclassifying by firing the neuron (finite copies of each test state):")
const COPIES = [2, 8, 32, 128, 512, 2048]
const REPS = 10
rng = MersenneTwister(SEED + 1)
firing = Dict()
for kind in (:margin, :square)
  m = models[kind]
  E, V = eigen(Symmetric(hamiltonian(m.terms, m.theta)))
  pops = populations(ds, test, V)
  firing[kind] = map(COPIES) do N
    mean(mean(classify_by_firing(kind, E, pops[:, j], T, N, rng) == ds.y[i]
              for (j, i) in enumerate(test)) for _ in 1:REPS)
  end
  @printf("  %-22s %s\n", NAME[kind],
          join((@sprintf("%d copies: %.3f", N, a) for (N, a) in zip(COPIES, firing[kind])), "   "))
end
writecsv(joinpath(RESULTS, "firing.csv"), ["copies", "accuracy_alg9_model", "accuracy_alg8_model"],
         [(N, firing[:margin][i], firing[:square][i]) for (i, N) in enumerate(COPIES)])

## 4. Figures (`plots.jl`)

The figures are drawn from the CSVs just written to `results/`. Each figure is shown inline and saved to `figures/`.

In [ ]:
# plots.jl — the figures, drawn from results/*.csv (run.jl calls this at the end).
#
#   julia 3_xx_xxx_grelu_minimal/plots.jl

using Plots, Statistics

const RES = joinpath(HERE, "results")
const FIG = mkpath(joinpath(HERE, "figures"))

"Read a CSV written by run.jl into named columns (numbers parsed where possible)."
function readcsv(name)
  lines = readlines(joinpath(RES, name))
  head = Symbol.(split(lines[1], ","))
  rows = [split(l, ",") for l in lines[2:end]]
  col(i) = (v = getindex.(rows, i); all(x -> tryparse(Float64, x) !== nothing, v) ? parse.(Float64, v) : String.(v))
  return NamedTuple{Tuple(head)}(Tuple(col(i) for i in eachindex(head)))
end

const LABEL = Dict("margin" => "Alg 9 (margin loss)", "square" => "Alg 8 (squared loss)")
const COLOR = Dict("margin" => :royalblue, "square" => :darkorange)
default(framestyle=:box, grid=false, dpi=200)

### Training loss and test accuracy.

In [ ]:
# Figure 1: training loss and test accuracy
h = readcsv("history.csv")
p1 = plot(xlabel="Adam step", ylabel="training loss", yscale=:log10, title="loss")
p2 = plot(xlabel="Adam step", ylabel="test accuracy", ylim=(0.4, 1.03), title="test accuracy", legend=:bottomright)
for k in ("margin", "square")
  s = h.model .== k
  plot!(p1, h.step[s], h.train_loss[s], lw=2, c=COLOR[k], label=LABEL[k])
  plot!(p2, h.step[s], h.test_accuracy[s], lw=2, c=COLOR[k], label=LABEL[k])
end
fig = plot(p1, p2, layout=(1, 2), size=(900, 350), margin=5Plots.mm)
savefig(fig, joinpath(FIG, "1_training.png"))
fig

### Learned weights by kind of term. The neurons compare ⟨ZZ⟩ with ⟨XX⟩ + ⟨YY⟩.

In [ ]:
# Figure 2: learned weights, averaged over each kind of term and scaled so the largest is 1
w = readcsv("weights.csv")
termkind(n) = n == "I" ? "I" : count(isletter, n) == 1 ? "Z" : string(n[1], n[1])   # "X3X4" -> "XX"
kinds = ["Z", "ZZ", "XX", "YY"]
p = plot(ylabel="mean weight (largest |.| = 1)", title="what the neuron learned", size=(600, 380),
         xticks=(1:4, kinds), ylim=(-1.12, 1.12), legend=:topleft)
for (off, k) in ((-0.18, "margin"), (0.18, "square"))
  s = w.model .== k
  m = [mean(w.theta[s][termkind.(w.term[s]) .== tk]) for tk in kinds]
  bar!(p, (1:4) .+ off, m ./ maximum(abs, m), bar_width=0.34, c=COLOR[k], label=LABEL[k])
end
hline!(p, [0], c=:black, lw=0.5, label="")
savefig(p, joinpath(FIG, "2_weights.png"))
p

### Decision score of every test state against temperature.

In [ ]:
# Figure 3: test scores by temperature (sign of the score = predicted label)
t = readcsv("test_scores.csv")
ps = map(("margin", "square")) do k
  s = t.model .== k
  q = plot(xscale=:log10, xlabel="kT", ylabel=k == "margin" ? "Tr[H ρ]" : "Tr[GReLU(H) ρ] − ½",
           title=LABEL[k] * "  (red XXX, grey XX)", titlefontsize=11, legend=false)
  for (y, c, lab) in ((-1, :gray40, "XX"), (1, :crimson, "XXX"))
    r = s .& (t.y .== y)
    scatter!(q, t.kT[r] .* exp.(0.04 .* randn(count(r))), t.score[r], ms=3, msw=0, alpha=0.7, c=c, label=lab)
  end
  hline!(q, [0], ls=:dash, c=:black, label="")
end
fig = plot(ps..., layout=(1, 2), size=(900, 350), margin=5Plots.mm)
savefig(fig, joinpath(FIG, "3_test_scores.png"))
fig

### Classifying by firing: accuracy against copies.

In [ ]:
# Figure 4: classifying by firing (Algorithm 5)
f = readcsv("firing.csv")
p = plot(xscale=:log2, xlabel="copies of each test state", ylabel="test accuracy", ylim=(0.45, 1.03),
         title="classifying by firing the neuron (Algorithm 5)", size=(600, 380), legend=:bottomright)
plot!(p, f.copies, f.accuracy_alg9_model, lw=2, marker=:circle, c=COLOR["margin"], label=LABEL["margin"] * " model")
plot!(p, f.copies, f.accuracy_alg8_model, lw=2, marker=:circle, c=COLOR["square"], label=LABEL["square"] * " model")
hline!(p, [0.5], ls=:dash, c=:gray, label="chance")
savefig(p, joinpath(FIG, "4_firing.png"))
p

## 5. Results and conclusions

- **Exact test accuracy:** both neurons classify all 160 test states correctly. ‖θ‖₁ is about 2.4 for the Algorithm 9 model and about 33 for the Algorithm 8 model.
- **Firing:** the Algorithm 8 model needs roughly four times fewer copies than the Algorithm 9 model for the same accuracy.
- **Why the task is easy:** XXX chains are rotation-symmetric, so ⟨ZZ⟩ = ⟨XX⟩ = ⟨YY⟩. In XX chains ⟨ZZ⟩ is much weaker. One linear combination of nearest-neighbour correlators therefore separates the classes.
- **More:** see [REPORT.md](REPORT.md), and folder 2 for shot noise, cross-validation, the ITensor qumode and the classical controls.